# Notebook 04: Data Quality Analysis

## 📚 Historical Context

**Timeline**: The Data Quality Crisis (2015-2024)

**The Wake-Up Call (2015-2018)**:
- **ImageNet**: Found 6% mislabeled images after years of use
- **CIFAR-10/100**: Estimated 3.4% label errors
- **Penn Treebank**: Inconsistent annotations discovered
- **Realization**: "More data" ≠ "better data"

**The Problems Discovered**:
- **Duplicates**: Test set samples in training set → inflated metrics
- **Label noise**: Human annotators disagree or make mistakes
- **PII leakage**: Personal information in training data
- **Toxicity**: Hate speech, profanity in web-scraped corpora
- **Data contamination**: Test sets leaked into training (GPT-3 controversy)

**The Evolution**:
- **2019**: Cleanlab - automated label error detection
- **2020**: Datasheets for Datasets (documentation standards)
- **2021**: Data-centric AI movement (Andrew Ng)
- **2022**: Near-duplicate detection becomes standard
- **2023**: EU AI Act requires data quality documentation

**Modern Reality**:
- **10-30% improvement** possible with same model + cleaner data
- **Data quality > Data quantity** (beyond a threshold)
- **Mandatory checks**: Duplicates, PII, toxicity, leakage

## 🎯 What You'll Learn

1. **Exploratory Data Analysis (EDA)** - Understanding your data
2. **Length Distribution** - Sequence length patterns
3. **Duplicate Detection** - Exact and near-duplicates (SimHash/MinHash)
4. **Outlier Detection** - Finding anomalous samples
5. **Label Consistency** - Checking classification labels
6. **Text Quality Metrics** - Readability, perplexity
7. **Toxic Content Detection** - Identifying harmful text
8. **PII Detection** - Finding personal information
9. **Data Leakage Detection** - Train/test contamination

## 💡 Why This Matters

**Real Examples of Data Quality Failures:**

**GPT-3 (2020)**: Accused of training on test sets
- Suspiciously high benchmark scores
- Near-duplicate detection found contamination
- Led to more rigorous deduplication standards

**Amazon Hiring AI (2018)**: Biased against women
- Trained on historical resumes (mostly male)
- Data reflected existing bias
- Penalized words like "women's chess club"

**Microsoft Tay (2016)**: Became toxic in 24 hours
- No toxicity filtering on training data
- Learned offensive language from users
- Shut down immediately

**Lesson**: Data quality analysis is not optional!

---

In [ ]:
# Import libraries
import os
import re
import json
import hashlib
from pathlib import Path
from collections import Counter, defaultdict
from typing import List, Dict, Set, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# NLP libraries
import nltk
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

# Set random seed
np.random.seed(42)

# Plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Data directory
data_dir = Path.cwd().parent / 'data'

print("Libraries imported successfully!")

## Part 1: Exploratory Data Analysis (EDA) for Text

### What to Analyze:

**1. Basic Statistics**
- Number of samples
- Number of classes (for classification)
- Class distribution

**2. Text Statistics**
- Character count distribution
- Word count distribution
- Token count distribution (after tokenization)
- Vocabulary size

**3. Quality Indicators**
- Empty or very short texts
- Very long texts (potential truncation)
- Special characters ratio
- Language detection (for multilingual)

### Why This Matters:
- **Imbalanced classes** → Need special handling
- **Length outliers** → Truncation strategy
- **Small vocabulary** → Might be too clean/synthetic
- **Large vocabulary** → Might need more cleaning

In [ ]:
print("=" * 60)
print("CREATING SAMPLE DATASET FOR ANALYSIS")
print("=" * 60)

# Create a realistic dataset with quality issues
np.random.seed(42)

sample_texts = [
    # Normal samples
    "This is a great product! I love it.",
    "Terrible quality, complete waste of money.",
    "It's okay, nothing special but works fine.",
    "Amazing service, highly recommended to everyone!",
    "Poor customer support, very disappointed.",
    
    # Duplicates (exact)
    "This is a great product! I love it.",  # Exact duplicate
    "This is a great product! I love it.",  # Another duplicate
    
    # Near-duplicates
    "This is a great product! I love it so much.",  # Near duplicate
    "This is a great product!! I love it.",  # Near duplicate (punctuation)
    
    # Quality issues
    "",  # Empty
    "x",  # Too short
    "Good " * 200,  # Too long (repetitive)
    
    # Potential PII
    "Contact me at john.doe@email.com or call 555-123-4567.",
    "My SSN is 123-45-6789 and I live at 123 Main St.",
    
    # Potential toxicity
    "This is garbage and you are an idiot.",
    "Stupid product, hate it.",
    
    # Outliers (different language/encoding)
    "Ceci est un texte en français.",  # French
    "これは日本語のテキストです。",  # Japanese
    
    # More normal samples
    "Fast delivery, product as described.",
    "Not worth the price, expected better quality.",
]

# Labels (intentionally include some inconsistencies)
labels = [
    'positive', 'negative', 'neutral', 'positive', 'negative',
    'positive', 'positive',  # Duplicates
    'positive', 'positive',  # Near-duplicates
    'neutral', 'neutral', 'positive',  # Quality issues
    'neutral', 'negative',  # PII
    'negative', 'negative',  # Toxic
    'positive', 'neutral',  # Different language
    'positive', 'negative',  # Normal
]

# Create dataset
dataset = Dataset.from_dict({
    'text': sample_texts,
    'label': labels
})

print(f"\nCreated dataset with {len(dataset)} samples")
print(f"Labels: {set(labels)}")

In [ ]:
print("=" * 60)
print("EXPLORATORY DATA ANALYSIS (EDA)")
print("=" * 60)

# 1. Basic Statistics
print("\n1. BASIC STATISTICS:")
print("-" * 40)
print(f"Total samples: {len(dataset)}")
print(f"Number of classes: {len(set(labels))}")
print(f"Class distribution: {Counter(labels)}")

# 2. Text Length Statistics
print("\n2. TEXT LENGTH STATISTICS:")
print("-" * 40)

# Calculate lengths
char_lengths = [len(text) for text in sample_texts]
word_lengths = [len(text.split()) for text in sample_texts]

print(f"\nCharacter lengths:")
print(f"  Min:    {min(char_lengths)}")
print(f"  Max:    {max(char_lengths)}")
print(f"  Mean:   {np.mean(char_lengths):.2f}")
print(f"  Median: {np.median(char_lengths):.2f}")
print(f"  Std:    {np.std(char_lengths):.2f}")

print(f"\nWord lengths:")
print(f"  Min:    {min(word_lengths)}")
print(f"  Max:    {max(word_lengths)}")
print(f"  Mean:   {np.mean(word_lengths):.2f}")
print(f"  Median: {np.median(word_lengths):.2f}")
print(f"  Std:    {np.std(word_lengths):.2f}")

# 3. Vocabulary Analysis
print("\n3. VOCABULARY ANALYSIS:")
print("-" * 40)

# Get all words
all_words = []
for text in sample_texts:
    all_words.extend(text.lower().split())

word_freq = Counter(all_words)
print(f"Total words: {len(all_words)}")
print(f"Unique words: {len(word_freq)}")
print(f"\nTop 10 most common words:")
for word, count in word_freq.most_common(10):
    print(f"  '{word}': {count}")

# 4. Visualizations
print("\n4. VISUALIZATIONS:")
print("-" * 40)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Plot 1: Class distribution
label_counts = Counter(labels)
axes[0, 0].bar(label_counts.keys(), label_counts.values(), alpha=0.7, color='steelblue')
axes[0, 0].set_title('Class Distribution')
axes[0, 0].set_ylabel('Count')
axes[0, 0].grid(axis='y', alpha=0.3)

# Plot 2: Character length distribution
axes[0, 1].hist(char_lengths, bins=20, alpha=0.7, color='coral', edgecolor='black')
axes[0, 1].set_title('Character Length Distribution')
axes[0, 1].set_xlabel('Characters')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].grid(axis='y', alpha=0.3)

# Plot 3: Word length distribution
axes[0, 2].hist(word_lengths, bins=20, alpha=0.7, color='green', edgecolor='black')
axes[0, 2].set_title('Word Length Distribution')
axes[0, 2].set_xlabel('Words')
axes[0, 2].set_ylabel('Frequency')
axes[0, 2].grid(axis='y', alpha=0.3)

# Plot 4: Box plot of character lengths by label
data_by_label = defaultdict(list)
for text, label in zip(sample_texts, labels):
    data_by_label[label].append(len(text))

axes[1, 0].boxplot(data_by_label.values(), labels=data_by_label.keys())
axes[1, 0].set_title('Character Length by Label')
axes[1, 0].set_ylabel('Characters')
axes[1, 0].grid(axis='y', alpha=0.3)

# Plot 5: Top words
top_words = word_freq.most_common(10)
words, counts = zip(*top_words)
axes[1, 1].barh(words, counts, alpha=0.7, color='purple')
axes[1, 1].set_title('Top 10 Most Common Words')
axes[1, 1].set_xlabel('Frequency')
axes[1, 1].grid(axis='x', alpha=0.3)

# Plot 6: Cumulative word frequency
sorted_freqs = sorted(word_freq.values(), reverse=True)
cumsum = np.cumsum(sorted_freqs)
axes[1, 2].plot(range(len(cumsum)), cumsum / cumsum[-1] * 100)
axes[1, 2].set_title('Cumulative Word Frequency')
axes[1, 2].set_xlabel('Number of Unique Words')
axes[1, 2].set_ylabel('Cumulative Frequency (%)')
axes[1, 2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ EDA complete! Check visualizations above.")
print("=" * 60)

## Part 2: Duplicate Detection

### Why Duplicates Matter:

**1. Data Leakage**
- Same sample in train and test → inflated metrics
- Model memorizes, doesn't generalize

**2. Wasted Computation**
- Training on duplicates wastes GPU time
- No new information

**3. Biased Learning**
- Overrepresented samples get more weight
- Model overfits to duplicates

### Detection Methods:

**1. Exact Duplicates**
- Hash text and compare
- O(n) with hash table
- Misses near-duplicates

**2. Near-Duplicates (SimHash)**
- Locality-sensitive hashing
- Catches similar texts
- Fast: O(n)

**3. Near-Duplicates (MinHash)**
- Jaccard similarity estimation
- Good for set overlap
- Used by Common Crawl

**4. Semantic Duplicates**
- Embedding similarity (cosine)
- Catches paraphrases
- Slow: O(n²) naive, O(n log n) with ANN

In [ ]:
print("=" * 60)
print("EXACT DUPLICATE DETECTION")
print("=" * 60)

def find_exact_duplicates(texts: List[str]) -> Dict[str, List[int]]:
    """
    Find exact duplicate texts using hashing.
    
    Args:
        texts: List of text strings
    
    Returns:
        Dictionary mapping text hash to list of indices
    """
    hash_to_indices = defaultdict(list)
    
    for idx, text in enumerate(texts):
        # Normalize text (lowercase, strip whitespace)
        normalized = text.lower().strip()
        
        # Create hash
        text_hash = hashlib.md5(normalized.encode()).hexdigest()
        
        hash_to_indices[text_hash].append(idx)
    
    # Return only duplicates (hash appears more than once)
    duplicates = {h: indices for h, indices in hash_to_indices.items() if len(indices) > 1}
    
    return duplicates

# Find exact duplicates
exact_dupes = find_exact_duplicates(sample_texts)

print(f"\nFound {len(exact_dupes)} groups of exact duplicates:")
print("-" * 40)

for i, (text_hash, indices) in enumerate(exact_dupes.items(), 1):
    print(f"\nGroup {i}: {len(indices)} copies at indices {indices}")
    print(f"Text: '{sample_texts[indices[0]][:60]}...'")

# Calculate statistics
total_duplicate_samples = sum(len(indices) - 1 for indices in exact_dupes.values())
duplicate_percentage = (total_duplicate_samples / len(sample_texts)) * 100

print(f"\n" + "=" * 60)
print(f"Total duplicate samples: {total_duplicate_samples}")
print(f"Percentage of dataset: {duplicate_percentage:.1f}%")
print(f"Unique samples: {len(sample_texts) - total_duplicate_samples}")
print("=" * 60)

In [ ]:
print("=" * 60)
print("NEAR-DUPLICATE DETECTION (SIMHASH)")
print("=" * 60)

class SimHash:
    """
    Simple SimHash implementation for near-duplicate detection.
    
    SimHash is a locality-sensitive hash that maps similar texts
    to similar hash values (small Hamming distance).
    """
    
    def __init__(self, hash_bits=64):
        self.hash_bits = hash_bits
    
    def _hash_token(self, token: str) -> int:
        """Hash a single token to an integer."""
        return int(hashlib.md5(token.encode()).hexdigest(), 16)
    
    def compute_hash(self, text: str) -> int:
        """
        Compute SimHash for a text.
        
        Algorithm:
        1. Tokenize text into words
        2. Hash each word to an integer
        3. For each bit position, accumulate +1 or -1 based on word hashes
        4. Final hash: 1 if accumulator > 0, else 0
        """
        # Tokenize (simple word split)
        tokens = text.lower().split()
        
        # Initialize accumulator for each bit
        v = [0] * self.hash_bits
        
        # For each token
        for token in tokens:
            # Hash token
            token_hash = self._hash_token(token)
            
            # For each bit position
            for i in range(self.hash_bits):
                # Check if bit i is set in token_hash
                if token_hash & (1 << i):
                    v[i] += 1  # Bit is 1
                else:
                    v[i] -= 1  # Bit is 0
        
        # Create final hash: 1 if v[i] > 0, else 0
        simhash = 0
        for i in range(self.hash_bits):
            if v[i] > 0:
                simhash |= (1 << i)
        
        return simhash
    
    def hamming_distance(self, hash1: int, hash2: int) -> int:
        """Compute Hamming distance between two hashes."""
        x = hash1 ^ hash2  # XOR to find differing bits
        
        # Count number of 1s (Brian Kernighan's algorithm)
        distance = 0
        while x:
            distance += 1
            x &= x - 1  # Remove rightmost 1
        
        return distance

# Find near-duplicates using SimHash
simhash = SimHash(hash_bits=64)

# Compute hashes for all texts
hashes = [simhash.compute_hash(text) for text in sample_texts]

print("\nComputed SimHashes for all texts")
print("Finding near-duplicates (Hamming distance ≤ 3)...\n")

# Find near-duplicates (small Hamming distance)
threshold = 3  # Maximum Hamming distance for near-duplicates
near_dupes = []

for i in range(len(hashes)):
    for j in range(i + 1, len(hashes)):
        distance = simhash.hamming_distance(hashes[i], hashes[j])
        
        if distance <= threshold:
            near_dupes.append((i, j, distance))

print(f"Found {len(near_dupes)} pairs of near-duplicates:")
print("-" * 40)

for idx1, idx2, distance in near_dupes:
    print(f"\nIndices: {idx1} ↔ {idx2} (Hamming distance: {distance})")
    print(f"Text 1: '{sample_texts[idx1][:60]}...'")
    print(f"Text 2: '{sample_texts[idx2][:60]}...'")

print("\n" + "=" * 60)
print("INTERPRETATION")
print("=" * 60)
print("SimHash finds texts that are similar but not identical.")
print("\nHamming distance interpretation:")
print("  0:     Exact duplicate (or very close)")
print("  1-3:   Near-duplicate (minor differences)")
print("  4-10:  Similar content")
print("  >10:   Different content")
print("\nTypical threshold: 3-5 for 64-bit hash")
print("=" * 60)

## Part 3: Outlier Detection

### Types of Outliers:

**1. Length Outliers**
- Too short: empty, single character
- Too long: might be concatenated documents

**2. Statistical Outliers**
- Unusual token distributions
- Repetitive patterns

**3. Language/Encoding Outliers**
- Different language
- Corrupted encoding
- Special characters

### Detection Methods:

**1. Z-score**
- How many standard deviations from mean
- |z| > 3 is typically an outlier

**2. IQR (Interquartile Range)**
- Values outside [Q1 - 1.5*IQR, Q3 + 1.5*IQR]
- More robust to extreme outliers

**3. Isolation Forest**
- ML-based anomaly detection
- Good for multivariate outliers

In [ ]:
print("=" * 60)
print("OUTLIER DETECTION")
print("=" * 60)

# Calculate features for outlier detection
features = []
for text in sample_texts:
    char_count = len(text)
    word_count = len(text.split())
    
    # Avoid division by zero
    avg_word_length = char_count / max(word_count, 1)
    
    # Special character ratio
    special_chars = sum(1 for c in text if not c.isalnum() and not c.isspace())
    special_ratio = special_chars / max(char_count, 1)
    
    # Digit ratio
    digits = sum(1 for c in text if c.isdigit())
    digit_ratio = digits / max(char_count, 1)
    
    # Uppercase ratio
    uppercase = sum(1 for c in text if c.isupper())
    uppercase_ratio = uppercase / max(char_count, 1)
    
    features.append({
        'index': len(features),
        'char_count': char_count,
        'word_count': word_count,
        'avg_word_length': avg_word_length,
        'special_ratio': special_ratio,
        'digit_ratio': digit_ratio,
        'uppercase_ratio': uppercase_ratio,
        'text_preview': text[:50] + '...' if len(text) > 50 else text
    })

df_features = pd.DataFrame(features)

# Method 1: Z-score for length outliers
print("\n1. LENGTH OUTLIERS (Z-score method):")
print("-" * 40)

mean_length = df_features['char_count'].mean()
std_length = df_features['char_count'].std()

# Calculate z-scores
df_features['z_score'] = (df_features['char_count'] - mean_length) / std_length

# Find outliers (|z| > 2)
length_outliers = df_features[df_features['z_score'].abs() > 2]

print(f"\nFound {len(length_outliers)} length outliers:")
for _, row in length_outliers.iterrows():
    print(f"\nIndex {row['index']}:")
    print(f"  Length: {row['char_count']} chars (z-score: {row['z_score']:.2f})")
    print(f"  Text: '{row['text_preview']}'")

# Method 2: IQR for all features
print("\n\n2. MULTIVARIATE OUTLIERS (IQR method):")
print("-" * 40)

def find_outliers_iqr(series, multiplier=1.5):
    """Find outliers using IQR method."""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR
    
    return (series < lower_bound) | (series > upper_bound)

# Check multiple features
outlier_features = ['special_ratio', 'digit_ratio', 'uppercase_ratio']
outlier_mask = pd.Series([False] * len(df_features))

for feature in outlier_features:
    outlier_mask |= find_outliers_iqr(df_features[feature])

multivar_outliers = df_features[outlier_mask]

print(f"\nFound {len(multivar_outliers)} multivariate outliers:")
for _, row in multivar_outliers.iterrows():
    print(f"\nIndex {row['index']}:")
    print(f"  Special chars: {row['special_ratio']:.3f}")
    print(f"  Digits:        {row['digit_ratio']:.3f}")
    print(f"  Uppercase:     {row['uppercase_ratio']:.3f}")
    print(f"  Text: '{row['text_preview']}'")

# Visualize outliers
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Character count with outliers marked
axes[0, 0].scatter(range(len(df_features)), df_features['char_count'], alpha=0.6)
axes[0, 0].scatter(length_outliers['index'], length_outliers['char_count'], 
                   color='red', s=100, marker='x', label='Outliers')
axes[0, 0].set_title('Character Count (Z-score outliers marked)')
axes[0, 0].set_xlabel('Sample Index')
axes[0, 0].set_ylabel('Character Count')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Plot 2: Box plot of character count
axes[0, 1].boxplot(df_features['char_count'])
axes[0, 1].set_title('Character Count Distribution')
axes[0, 1].set_ylabel('Character Count')
axes[0, 1].grid(axis='y', alpha=0.3)

# Plot 3: Special character ratio
axes[1, 0].scatter(range(len(df_features)), df_features['special_ratio'], alpha=0.6)
axes[1, 0].scatter(multivar_outliers['index'], multivar_outliers['special_ratio'],
                   color='red', s=100, marker='x', label='Outliers')
axes[1, 0].set_title('Special Character Ratio')
axes[1, 0].set_xlabel('Sample Index')
axes[1, 0].set_ylabel('Ratio')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Plot 4: Digit ratio
axes[1, 1].scatter(range(len(df_features)), df_features['digit_ratio'], alpha=0.6)
axes[1, 1].scatter(multivar_outliers['index'], multivar_outliers['digit_ratio'],
                   color='red', s=100, marker='x', label='Outliers')
axes[1, 1].set_title('Digit Ratio')
axes[1, 1].set_xlabel('Sample Index')
axes[1, 1].set_ylabel('Ratio')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("OUTLIER HANDLING STRATEGIES")
print("=" * 60)
print("1. Remove: For clear data quality issues")
print("2. Cap: Set to maximum/minimum threshold")
print("3. Investigate: Manual review for edge cases")
print("4. Keep: If they represent valid rare cases")
print("=" * 60)

## Part 4: PII (Personally Identifiable Information) Detection

### Why PII Matters:

**Legal**:
- GDPR (EU): Fines up to €20M or 4% of revenue
- CCPA (California): $7,500 per violation
- HIPAA (Healthcare): Up to $50,000 per record

**Ethical**:
- Privacy violation
- Model can memorize and leak PII
- Reputational damage

### Common PII Types:

| Type | Example | Regex Pattern |
|------|---------|---------------|
| Email | john@example.com | `\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z\|a-z]{2,}\b` |
| Phone (US) | 555-123-4567 | `\b\d{3}[-.]?\d{3}[-.]?\d{4}\b` |
| SSN (US) | 123-45-6789 | `\b\d{3}-\d{2}-\d{4}\b` |
| Credit Card | 4532-1234-5678-9010 | `\b\d{4}[-\s]?\d{4}[-\s]?\d{4}[-\s]?\d{4}\b` |
| IP Address | 192.168.1.1 | `\b(?:[0-9]{1,3}\.){3}[0-9]{1,3}\b` |
| URL | https://example.com | `https?://[^\s]+` |

### Detection Strategy:
1. Regex patterns (fast, rule-based)
2. NER models (more accurate, slower)
3. Combination of both

In [ ]:
print("=" * 60)
print("PII DETECTION")
print("=" * 60)

class PIIDetector:
    """
    Detect Personally Identifiable Information (PII) in text.
    
    Uses regex patterns for common PII types.
    """
    
    def __init__(self):
        # Define regex patterns for common PII
        self.patterns = {
            'email': r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',
            'phone_us': r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b',
            'ssn': r'\b\d{3}-\d{2}-\d{4}\b',
            'credit_card': r'\b\d{4}[-\s]?\d{4}[-\s]?\d{4}[-\s]?\d{4}\b',
            'ip_address': r'\b(?:[0-9]{1,3}\.){3}[0-9]{1,3}\b',
            'url': r'https?://[^\s]+',
        }
        
        # Compile patterns
        self.compiled_patterns = {
            name: re.compile(pattern, re.IGNORECASE)
            for name, pattern in self.patterns.items()
        }
    
    def detect(self, text: str) -> Dict[str, List[str]]:
        """
        Detect PII in text.
        
        Returns:
            Dictionary mapping PII type to list of matches
        """
        results = {}
        
        for name, pattern in self.compiled_patterns.items():
            matches = pattern.findall(text)
            if matches:
                results[name] = matches
        
        return results
    
    def anonymize(self, text: str) -> str:
        """
        Replace PII with placeholders.
        """
        anonymized = text
        
        # Replace each PII type with placeholder
        for name, pattern in self.compiled_patterns.items():
            placeholder = f'<{name.upper()}>'
            anonymized = pattern.sub(placeholder, anonymized)
        
        return anonymized

# Test PII detection
pii_detector = PIIDetector()

print("\nScanning dataset for PII...\n")
print("-" * 60)

pii_found = False
for idx, text in enumerate(sample_texts):
    pii = pii_detector.detect(text)
    
    if pii:
        pii_found = True
        print(f"\n⚠️  PII DETECTED in sample {idx}:")
        print(f"Text: '{text}'")
        print(f"\nPII found:")
        for pii_type, matches in pii.items():
            print(f"  {pii_type}: {matches}")
        
        # Show anonymized version
        anonymized = pii_detector.anonymize(text)
        print(f"\nAnonymized: '{anonymized}'")
        print("-" * 60)

if not pii_found:
    print("✓ No PII detected in dataset")

print("\n" + "=" * 60)
print("PII HANDLING BEST PRACTICES")
print("=" * 60)
print("1. ALWAYS scan datasets before training")
print("2. Remove or anonymize PII")
print("3. Document PII handling procedures")
print("4. Consider using specialized NER models for better detection")
print("5. Be extra careful with web-scraped data")
print("6. Legal: Check GDPR, CCPA, HIPAA requirements")
print("=" * 60)

## Part 5: Label Consistency Checking (for Classification)

### Why Check Labels?

**Human Error**:
- Annotators make mistakes
- Ambiguous cases labeled inconsistently
- Fatigue in long annotation sessions

**Impact**:
- Model learns noise, not signal
- Lower accuracy ceiling
- Confusion in decision boundaries

### Detection Methods:

**1. Duplicate text, different labels**
- Same text shouldn't have different labels
- Clear indicator of inconsistency

**2. Near-duplicate text, different labels**
- Very similar text with different labels
- Might be inconsistent or genuinely different

**3. Confident mispredictions**
- Train model, find high-confidence errors
- Model might be right, label might be wrong

In [ ]:
print("=" * 60)
print("LABEL CONSISTENCY CHECKING")
print("=" * 60)

def check_label_consistency(texts: List[str], labels: List[str]) -> Dict:
    """
    Check for label inconsistencies.
    
    Returns:
        Dictionary with inconsistency information
    """
    # Create text -> labels mapping
    text_to_labels = defaultdict(set)
    text_to_indices = defaultdict(list)
    
    for idx, (text, label) in enumerate(zip(texts, labels)):
        # Normalize text
        normalized = text.lower().strip()
        
        text_to_labels[normalized].add(label)
        text_to_indices[normalized].append(idx)
    
    # Find inconsistencies (same text, multiple labels)
    inconsistencies = []
    for text, label_set in text_to_labels.items():
        if len(label_set) > 1:
            indices = text_to_indices[text]
            inconsistencies.append({
                'text': text,
                'labels': label_set,
                'indices': indices,
                'count': len(indices)
            })
    
    return {
        'inconsistencies': inconsistencies,
        'num_inconsistent_groups': len(inconsistencies),
        'total_inconsistent_samples': sum(len(inc['indices']) for inc in inconsistencies)
    }

# Check label consistency
result = check_label_consistency(sample_texts, labels)

print(f"\nFound {result['num_inconsistent_groups']} groups with label inconsistencies:")
print(f"Total inconsistent samples: {result['total_inconsistent_samples']}\n")
print("-" * 60)

for i, inc in enumerate(result['inconsistencies'], 1):
    print(f"\nInconsistency Group {i}:")
    print(f"Text: '{inc['text'][:60]}...'")
    print(f"Found at indices: {inc['indices']}")
    print(f"Different labels: {inc['labels']}")
    print(f"\nDetails:")
    for idx in inc['indices']:
        print(f"  Index {idx}: label = '{labels[idx]}'")
    print("-" * 60)

# Calculate label distribution
print("\n" + "=" * 60)
print("LABEL DISTRIBUTION ANALYSIS")
print("=" * 60)

label_counts = Counter(labels)
total = len(labels)

print("\nClass distribution:")
for label, count in label_counts.most_common():
    percentage = (count / total) * 100
    print(f"  {label:10s}: {count:3d} samples ({percentage:5.1f}%)")

# Check for severe imbalance
max_count = max(label_counts.values())
min_count = min(label_counts.values())
imbalance_ratio = max_count / min_count

print(f"\nImbalance ratio: {imbalance_ratio:.2f}:1")
if imbalance_ratio > 3:
    print("⚠️  WARNING: Severe class imbalance detected!")
    print("Consider: stratified sampling, class weights, or resampling")

# Visualize
plt.figure(figsize=(10, 5))
plt.bar(label_counts.keys(), label_counts.values(), alpha=0.7, color='steelblue')
plt.xlabel('Label')
plt.ylabel('Count')
plt.title('Label Distribution')
plt.grid(axis='y', alpha=0.3)
plt.show()

print("\n" + "=" * 60)
print("RECOMMENDATIONS")
print("=" * 60)
print("1. Review all inconsistent labels manually")
print("2. Establish clear labeling guidelines")
print("3. Use multiple annotators with agreement metrics")
print("4. Consider removing ambiguous samples")
print("5. Document label decisions for reproducibility")
print("=" * 60)

## 📊 Summary and Key Takeaways

### What We Learned:

**1. Exploratory Data Analysis**
- Basic statistics (count, distribution)
- Length analysis (characters, words, tokens)
- Vocabulary analysis
- Visualizations for understanding

**2. Duplicate Detection**
- Exact duplicates: Fast hash-based detection
- Near-duplicates: SimHash (Hamming distance)
- Alternative: MinHash (Jaccard similarity)
- Impact: Data leakage, wasted computation, biased learning

**3. Outlier Detection**
- Length outliers (Z-score, IQR)
- Statistical outliers (multivariate analysis)
- Language/encoding outliers
- Handling: Remove, cap, investigate, or keep

**4. PII Detection**
- Regex patterns for common PII types
- Anonymization strategies
- Legal implications (GDPR, CCPA, HIPAA)
- Always scan before training!

**5. Label Consistency**
- Same text, different labels → inconsistency
- Class imbalance detection
- Impact on model performance
- Need for clear guidelines

### Data Quality Checklist:

**Before Fine-Tuning:**
```python
# 1. EDA
✓ Analyze length distributions
✓ Check class balance
✓ Visualize key statistics

# 2. Duplicates
✓ Remove exact duplicates
✓ Check for near-duplicates
✓ Verify no train/test overlap

# 3. Outliers
✓ Detect length outliers
✓ Check for encoding issues
✓ Handle or remove

# 4. PII
✓ Scan for emails, phones, SSN, etc.
✓ Anonymize or remove
✓ Document handling

# 5. Labels
✓ Check consistency
✓ Verify class balance
✓ Review ambiguous cases
```

### Impact of Data Quality:

| Issue | Impact | Detection | Solution |
|-------|--------|-----------|----------|
| Duplicates | Data leakage | Hash/SimHash | Remove |
| Outliers | Poor generalization | Z-score/IQR | Remove/cap |
| PII | Legal/privacy risk | Regex/NER | Anonymize |
| Label errors | Lower accuracy | Consistency check | Re-label |
| Imbalance | Biased predictions | Distribution analysis | Resample/weight |

### Real-World Statistics:

**From Research**:
- ImageNet: ~6% label errors
- CIFAR-10: ~3.4% label errors
- Common Crawl: ~10-30% near-duplicates

**Performance Impact**:
- Removing duplicates: +5-15% validation accuracy
- Fixing label errors: +3-10% accuracy
- PII removal: No accuracy change, but legally required

### Tools for Production:

**Open Source**:
- Cleanlab: Automated label error detection
- Presidio: Microsoft's PII detection
- Datasketch: MinHash/LSH for near-duplicates

**Commercial**:
- Scale AI: Data labeling + quality
- Snorkel: Programmatic labeling
- Great Expectations: Data validation

### Next Notebook:
**05_handling_class_imbalance.ipynb**

We'll explore:
- Class distribution visualization
- Random oversampling
- Random undersampling
- SMOTE for text (limitations)
- Class weighting
- Stratified sampling
- Comparing all methods
- When to use which technique

---

**Historical Note**: The importance of data quality wasn't fully appreciated until 2018-2019 when several high-profile studies revealed significant label errors in benchmark datasets. This led to the "Data-Centric AI" movement championed by Andrew Ng in 2021, shifting focus from "better models" to "better data." Today, companies spend 60-80% of ML project time on data quality, compared to <20% in 2015. The lesson: **Clean data with simple models beats dirty data with complex models, every time.**